# DL — LSTM: OHLCV vs OHLCV + Indicators
### TCS Stock Price Prediction | Deep Learning — Long Short-Term Memory
**Target:** Predict next-day RETURN (converted back to price for all metrics & graphs)  
**Two Sets:** Set A (OHLCV, 5 features) · Set B (OHLCV + 9 Indicators, 14 features)  
**Flow:** EDA → Part A (OHLCV) → Compute Indicators → Part B (Indicators) → Compare A vs B


## Why LSTM for Stock Price Prediction?
> **LSTM (Long Short-Term Memory)** networks maintain a memory cell that selectively retains or forgets information over long sequences — ideal for time-series with long-range dependencies.  
> - Hochreiter & Schmidhuber (1997) introduced LSTM specifically to overcome vanishing gradients in RNNs.  
> - Fischer & Krauss (2018) showed LSTM outperforms DNN, RF and logistic regression on S&P 500 daily prediction.  
> - **Why RETURN not PRICE?** Predicting log-returns (or simple returns) is statistically cleaner: returns are approximately stationary, whereas raw prices have strong trends that can create misleading high R² due to autocorrelation.  
> - **StandardScaler** is preferred over MinMaxScaler for return-based targets because returns can be negative; StandardScaler handles this naturally without clipping.


In [ ]:
# Dependencies already installed in environment
# !pip install mplfinance scikit-learn -q
print('Dependencies ready!')


---
## Step 0: Reproducibility — Set All Random Seeds to 42
Setting `numpy`, `python random`, and `tensorflow` seeds to **42** for fully reproducible results.


In [ ]:
import os, random
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)

import numpy as np
np.random.seed(42)

import tensorflow as tf
tf.random.set_seed(42)

# Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All imports done! TensorFlow version:', tf.__version__)
print('All random seeds set to 42.')


---
## Step 1: Load TCS Dataset
Reading `TCS.csv` (skiprows=[1,2]), parsing date index, keeping OHLCV columns, dropping NaN rows.
